In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
data = pd.read_csv('HousingData.csv')

In [ ]:
data.head(10)

In [ ]:
data.info()

In [ ]:
data.duplicated().sum()

In [ ]:
data.isnull().sum()

In [ ]:
for col in ['CRIM','ZN','INDUS','CHAS','AGE','LSTAT']:
    data[col].fillna(data[col].median(), inplace=True)

In [ ]:
data.info()

In [ ]:
data.hist(figsize=(15,10), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,8))

sns.boxplot(data=data)

plt.xticks(rotation=90)

plt.show()

In [ ]:
corr = data.corr()

plt.figure(figsize=(14,12))

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
X = data.drop('MEDV', axis=1)
y = data['MEDV']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
lr = LinearRegression()

lr.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
y_pred_lr = lr.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_lr)
mse = mean_squared_error(y_test, y_pred_lr)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_lr)

print("Linear Regression")
print("-"*40)
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

In [ ]:
cv = cross_val_score(
    LinearRegression(),
    X,
    y,
    cv=5,
    scoring='r2'
)

print("Cross Validation Scores:")
print(cv)

print()

print("Mean R2:", cv.mean())

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=42)

dt.fit(X_train, y_train)

In [ ]:
y_pred_dt = dt.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_dt)
mse = mean_squared_error(y_test, y_pred_dt)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_dt)

print("Decision Tree")
print("-"*40)
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

In [ ]:
cv_dt = cross_val_score(
    DecisionTreeRegressor(random_state=42),
    X,
    y,
    cv=5,
    scoring='r2'
)

print(cv_dt)
print()

print("Mean R2:", cv_dt.mean())

In [ ]:
from sklearn.model_selection import GridSearchCV

param_dt = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_dt = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid=param_dt,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_dt.fit(X_train, y_train)

print("Best Parameters:", grid_dt.best_params_)
print("Best CV Score:", grid_dt.best_score_)

In [ ]:
best_dt = grid_dt.best_estimator_

y_pred_best_dt = best_dt.predict(X_test)

print("R2:", r2_score(y_test, y_pred_best_dt))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_best_dt)))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    random_state=42
)

rf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

y_pred_rf = rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_rf)
mse = mean_squared_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_rf)

print("Random Forest Regressor")
print("-"*40)
print(f"MAE : {mae:.4f}")
print(f"MSE : {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
from sklearn.model_selection import cross_val_score

cv_rf = cross_val_score(
    RandomForestRegressor(random_state=42),
    X,
    y,
    cv=5,
    scoring='r2'
)

print("Cross Validation Scores:")
print(cv_rf)

print()

print("Mean R²:", cv_rf.mean())

In [ ]:
from sklearn.model_selection import GridSearchCV

param_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_rf = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=param_rf,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("Best Parameters:")
print(grid_rf.best_params_)

print()

print("Best CV Score:")
print(grid_rf.best_score_)

In [ ]:
best_rf = grid_rf.best_estimator_

y_pred_best_rf = best_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_best_rf)
mse = mean_squared_error(y_test, y_pred_best_rf)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_best_rf)

print("Best Random Forest")
print("-"*40)
print(f"MAE : {mae:.4f}")
print(f"MSE : {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")